# Laerning Curves

there are two most import curves to monitor are:
1. loss curves - show how the model error(loss) changes over training steps or epochs

2. accuracy curves - show how the percentage of correct prediction over training steps or epochs

# Loss curves


The loss curve shows how the model’s error decreases over time. In a typical successful training run

High initial loss: The model starts without optimization, so predictions are initially poor

Decreasing loss: As training progresses, the loss should generally decrease

Convergence: Eventually, the loss stabilizes at a low value, indicating that the model has learned the patterns in the data


In [ ]:
# Example of tracking loss during training with the Trainer
from transformers import Trainer, TrainingArguments
import wandb

# Initialize Weights & Biases for experiment tracking
wandb.init(project="transformer-fine-tuning", name="bert-mrpc-analysis")

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    logging_steps=10,  # Log metrics every 10 steps
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    report_to="wandb",  # Send logs to Weights & Biases
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train and automatically log metrics
trainer.train()

# Accuracy Curves


The accuracy curve shows the percentage of correct predictions over time. Unlike loss curves, accuracy curves should generally increase as the model learns and can typically include more steps than the loss curve

Start low: Initial accuracy should be low, as the model has not yet learned the patterns in the data

Increase with training: Accuracy should generally improve as the model learns if it is able to learn the patterns in the data

May show plateaus: Accuracy often increases in discrete jumps rather than smoothly, as the model makes predictions that are close to the true label

# covergence

Convergence occurs when the model’s performance stabilizes and the loss and accuracy curves level off. This is a sign that the model has learned the patterns in the data and is ready to be used. In simple terms, we are aiming for the model to converge to a stable performance every time we train it.

Once models have converged, we can use them to make predictions on new data and refer to evaluation metrics to understand how well the model is performing.

# Intrepreting Learning Curves Patterns

## 1. Healthy Learning Curves

The loss can improve if the model’s output gets closer to the target, even if the final prediction is still incorrect. Accuracy, however, only improves when the prediction crosses the threshold to be correct.

Characteristics of healthy curves:

Smooth decline in loss: Both training and validation loss decrease steadily
Close training/validation performance: Small gap between training and validation metrics
Convergence: Curves level off, indicating the model has learned the patterns



During Trainig and After Training

During Training
During the training process (after you’ve hit trainer.train()), you can monitor these key indicators:

Loss convergence: Is the loss still decreasing or has it plateaued?

Overfitting signs: Is validation loss starting to increase while training loss decreases?

Learning rate: Are the curves too erratic (LR too high) or too flat (LR too low)?

Stability: Are there sudden spikes or drops that indicate problems?

After Training
After the training process is complete, you can analyze the complete curves to understand the model’s performance.


Final performance: Did the model reach acceptable performance levels?

Efficiency: Could the same performance be achieved with fewer epochs?

Generalization: How close are training and validation performance?

Trends: Would additional training likely improve performance?


# overfitting

it happens when the model learns too much from the training data and unable to generalize to different data

Symptoms:

Training loss continues to decrease while validation loss increases or plateaus.

Large gap between training and validation .

Training accuracy much higher than validation accuracy.

Solutions for overfitting:

Regularization: Add dropout, weight decay, or other regularization techniques

Early stopping: Stop training when validation performance stops improving

Data augmentation: Increase training data diversity

Reduce model complexity: Use a smaller model or fewer parameters

In [ ]:
# Example of detecting overfitting with early stopping
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    num_train_epochs=10,  # Set high, but we'll stop early
)

# Add early stopping to prevent overfitting
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Underfitting
 it occurs when the model is too simple to capture the underlying patterins in the data

"""
1. The model is too small or lacks capacity to learn the patterns
2. The learning rate is too low, causing slow learning
3. The dataset is too small or not representative of the problem
4. The model is not properly regularized

Symptoms:

1. Both training and validation loss remain high
2. Model performance plateaus early in training
3. Training accuracy is lower than expected


Solutions for underfitting:

1. Increase model capacity: Use a larger model or more parameters
2. Train longer: Increase the number of epochs
3. Adjust learning rate: Try different learning rates
4. Check data quality: Ensure your data is properly preprocessed


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    -num_train_epochs=5,
    +num_train_epochs=10,
)

# Erratic Learning Curves
Erratic learning curves occur when the model is not learning effectively. This can happen for several reasons:

1. The learning rate is too high, causing the model to overshoot the optimal parameters
2. The batch size is too small, causing the model to learn slowly
3. The model is not properly regularized, causing it to overfit to the training data
4. The dataset is not properly preprocessed, causing the model to learn from noise

Symptoms:

1. Frequent fluctuations in loss or accuracy
2. Curves show high variance or instability
3. Performance oscillates without clear trend


Solutions for erratic curves:

1. Lower learning rate: Reduce step size for more stable training
2. Increase batch size: Larger batches provide more stable gradients
3. Gradient clipping: Prevent exploding gradients
4. Better data preprocessing: Ensure consistent data quality

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    -learning_rate=1e-5,
    +learning_rate=1e-4,
    -per_device_train_batch_size=16,
    +per_device_train_batch_size=32,
)